<a href="https://colab.research.google.com/github/taylan-sen/CIS490b_computer_vision/blob/main/Motion_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem:** - how to detect/understand object motion from a series of images (i.e. a video)
<img src="https://nanonets.com/blog/content/images/2019/04/intro-1-2.gif">

* How can we more specifically pose the question?  
* What should be acceptable inputs?   
* How do we report a solution (what are outputs)?  
  * A grid of motion estimates (sparse optical flow).
  * A motion estimate for each pixel (dense optical flow).

<img src="https://nanonets.com/blog/content/images/2019/04/sparse-vs-dense.gif">




This notebook was based on these articles:[one](https://nanonets.com/blog/optical-flow/), [two](https://androidkt.com/how-to-capture-and-play-video-in-google-colab/)

The two cells below allow you to capture a video via Google colab (similar to how we used Code Snippets for taking pictures in the past). Take a short video with some type of motion - Ex: move your hand or head.

In [ ]:
from IPython.display import display, Javascript,HTML
from google.colab.output import eval_js
from base64 import b64decode

def record_video(filename):
  js=Javascript("""
    async function recordVideo() {
      const options = { mimeType: "video/webm; codecs=vp9" };
      const div = document.createElement('div');
      const capture = document.createElement('button');
      const stopCapture = document.createElement("button");

      capture.textContent = "Start Recording";
      capture.style.background = "orange";
      capture.style.color = "white";

      stopCapture.textContent = "Stop Recording";
      stopCapture.style.background = "red";
      stopCapture.style.color = "white";
      div.appendChild(capture);

      const video = document.createElement('video');
      const recordingVid = document.createElement("video");
      video.style.display = 'block';

      const stream = await navigator.mediaDevices.getUserMedia({audio:true, video: true});

      let recorder = new MediaRecorder(stream, options);
      document.body.appendChild(div);
      div.appendChild(video);

      video.srcObject = stream;
      video.muted = true;

      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => {
        capture.onclick = resolve;
      });
      recorder.start();
      capture.replaceWith(stopCapture);

      await new Promise((resolve) => stopCapture.onclick = resolve);
      recorder.stop();
      let recData = await new Promise((resolve) => recorder.ondataavailable = resolve);
      let arrBuff = await recData.data.arrayBuffer();

      // stop the stream and remove the video element
      stream.getVideoTracks()[0].stop();
      div.remove();

      let binaryString = "";
      let bytes = new Uint8Array(arrBuff);
      bytes.forEach((byte) => {
        binaryString += String.fromCharCode(byte);
      })
    return btoa(binaryString);
    }
  """)
  try:
    display(js)
    data=eval_js('recordVideo({})')
    binary=b64decode(data)
    with open(filename,"wb") as video_file:
      video_file.write(binary)
    print(f"Finished recording video at:{filename}")
  except Exception as err:
    print(str(err))

In [ ]:
video_path = "test.mp4"
record_video(video_path)

<IPython.core.display.Javascript object>

NotSupportedError: MediaRecorder constructor: video/webm; codecs=vp9 indicates an unsupported codec


<hr>
Grayscale the video.

###Optical Flow steps

* Put video in a "good" format
* Identify/specify a set of (x,y) features to track.
* Find the displacement of each feature between successive frames:
  * Do a convolution/kernel "scan" of the feature pixels in each subsequent image.
  * Calculate the x and y displacement and record it for each feature.

In [ ]:
import cv2 as cv
import numpy as np

# Parameters for Shi-Tomasi corner detection
feature_params = dict(maxCorners = 300, qualityLevel = 0.2, minDistance = 2,
                      blockSize = 7)
# Parameters for Lucas-Kanade optical flow
lk_params = dict(winSize = (15,15), maxLevel = 2,
                 criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT,
                             10, 0.03))
# The video feed is read in as a VideoCapture object
#cap = cv.VideoCapture("shibuya.mp4")
cap = cv.VideoCapture("right.mp4")
# Variable for color to draw optical flow track
color = (0, 255, 0)
# ret = a boolean return value from getting the frame,
# first_frame = the first frame in the entire video sequence
ret, first_frame = cap.read()
outfile = cv.VideoWriter("output.avi",cv.VideoWriter_fourcc(*'MPEG'),
                         30,(first_frame.shape[1], first_frame.shape[0]))

# Converts frame to grayscale because we only need the luminance channel
# for detecting edges - less computationally expensive
prev_gray = cv.cvtColor(first_frame, cv.COLOR_BGR2GRAY)
# Finds the strongest corners in the first frame by Shi-Tomasi method
#  - we will track the optical flow for these corners
prev = cv.goodFeaturesToTrack(prev_gray, mask = None, **feature_params)
# Creates an image filled with zero intensities with the same dimensions
# as the frame - for later drawing purposes
mask = np.zeros_like(first_frame)

while(cap.isOpened()):
    # ret = a boolean return value from getting the frame,
    # frame = the current frame being projected in the video
    ret, frame = cap.read()
    if not ret:
      print('read empty, breaking')
      break
    # Converts each frame to grayscale
    # - we previously only converted the first frame to grayscale
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    # Calculates sparse optical flow by Lucas-Kanade method
    prev = cv.goodFeaturesToTrack(prev_gray, mask = None, **feature_params)
    next, status, error = cv.calcOpticalFlowPyrLK(prev_gray, gray, prev,
                                                  None, **lk_params)
    # Selects good feature points for previous position
    good_old = prev[status == 1].astype(int)
    # Selects good feature points for next position
    good_new = next[status == 1].astype(int)
    # Draws the optical flow tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        # Returns a contiguous flattened array as (x, y) coordinates for new point
        a, b = new.ravel()
        # Returns a contiguous flattened array as (x, y) coordinates for old point
        c, d = old.ravel()
        # Draws line between new and old position with green color and 2 thickness
        mask = cv.line(mask, (a, b), (c, d), color, 2)
        # Draws filled circle (thickness of -1) at new position with green color and radius of 3
        frame = cv.circle(frame, (a, b), 3, color, -1)
    # Overlays the optical flow tracks on the original frame
    output = cv.add(frame, mask)
    mask = np.zeros_like(first_frame)
    # Updates previous frame
    prev_gray = gray.copy()
    # Updates previous good feature points
    prev = good_new.reshape(-1, 1, 2)
    # Opens a new window and displays the output frame
    #cv.imshow("sparse optical flow", output)
    #cv2_imshow(output)
    outfile.write(output)
    # Frames are read by intervals of 10 milliseconds.
    # The programs breaks out of the while loop when the user presses the 'q' key
    if (cv.waitKey(10) & 0xFF == ord('q')):
        break
# The following frees up resources and closes all windows
cap.release()
outfile.release()
cv.destroyAllWindows()

read empty, breaking
